# The Single Neuron — Hands-on Tutorial

In this notebook you will build intuition for:
1. What a single neuron computes — and what controls its decision boundary
2. How the choice of activation changes behaviour and gradients
3. The gap between a clean toy problem and a realistic one
4. Why one neuron is not enough (XOR)

## Where this fits

| # | Topic | Slide deck | Notebook |
|---|---|---|---|
| **→ 1** | **Single neuron** | **`01_single_neuron.pdf`** | **`01_single_neuron.ipynb`** |
| 2 | Multilayer networks | `02_multilayer_networks.pdf` | `02_training_loop.ipynb` |
| 3 | Backpropagation | `03_backprop_training.pdf` | `03_backpropagation.ipynb` |
| 4 | Optimizers | `07_optimizers.pdf` *(new)* | `04_optimizers.ipynb` |
| 5 | CNNs | `04_cnns.pdf` | `05_cnns.ipynb` |
| 6 | Modern architectures | `05a_attention.pdf` + `05b_practical.pdf` | *(bonus: `bonus_generative_models.ipynb`)* |
| 7 | Bayesian inference | `06_bayesian_inference.pdf` | `06_bayesian_inference.ipynb` |

**Coming from:** You've worked with gradient descent on simple functions earlier in the school; now you'll watch it shape the decision boundary of a single neuron.

**Leading to:** Notebook 02 stacks neurons into layers and trains them end-to-end with a real loss and optimizer.

**If you skipped ahead:** If you know what a derivative is and can read `numpy`, you have enough to follow along.


In [1]:
# Run on Colab only — skip on JupyterHub (packages are pre-installed)
import sys
if 'google.colab' in sys.modules:
    %pip install -q ipywidgets

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, ToggleButtons

%matplotlib inline

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Shared slider range used in Parts 1 and 3
kw = dict(min=-6.0, max=6.0, step=0.1)

---

## The mathematics of a single neuron

### The neuron equation

$$\hat{y} = \sigma(z), \qquad z = \mathbf{w} \cdot \mathbf{x} + b$$

| Symbol | Meaning |
|--------|---------|
| $\mathbf{x} \in \mathbb{R}^n$ | Input feature vector — e.g. $(\text{dip depth},\, \text{dip duration})$ |
| $\mathbf{w} \in \mathbb{R}^n$ | Weight vector — how much each feature contributes |
| $b \in \mathbb{R}$ | Bias — shifts the threshold independently of the input |
| $z \in \mathbb{R}$ | Pre-activation — the raw linear combination |
| $\sigma$ | Activation function — squashes $z$ into a bounded range |
| $\hat{y} \in (0,1)$ | Predicted probability of the positive class |

**The decision boundary** is the set of points where $z = 0$, i.e. $\mathbf{w}\cdot\mathbf{x} + b = 0$. On one side $\hat{y} > 0.5$, on the other $\hat{y} < 0.5$.

### The sigmoid activation

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \sigma'(z) = \sigma(z)\,\bigl(1 - \sigma(z)\bigr)$$

The derivative $\sigma'(z)$ will reappear in backpropagation — it is the channel through which the learning signal flows. Notice it is maximised at $z=0$ and approaches zero at both extremes.

In [ ]:
# ── Implement σ(z) and σ′(z) then plot them ───────────────────────────────────

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

z = np.linspace(-6, 6, 400)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: σ(z)
axes[0].plot(z, sigmoid(z), lw=2.5, color='steelblue')
axes[0].axhline(0.5, color='gray', ls='--', lw=1, label='$\sigma(z) = 0.5$  (decision boundary)')
axes[0].axvline(0,   color='gray', ls='--', lw=1)
axes[0].fill_between(z, sigmoid(z), 0.5, where=(z > 0), alpha=0.12, color='green',  label='predicts +')
axes[0].fill_between(z, sigmoid(z), 0.5, where=(z < 0), alpha=0.12, color='tomato', label='predicts −')
axes[0].set_xlabel('$z = \mathbf{w}\cdot\mathbf{x} + b$')
axes[0].set_ylabel('$\sigma(z)$')
axes[0].set_title('Sigmoid output — predicted probability')
axes[0].legend()

# Right: σ′(z)
axes[1].plot(z, sigmoid_deriv(z), lw=2.5, color='darkorange', label="$\sigma'(z)$")
axes[1].axhline(0.25, color='gray', ls='--', lw=1, label='max = 0.25 at z=0')
axes[1].fill_between(z, sigmoid_deriv(z), where=(np.abs(z) > 3),
                     alpha=0.25, color='red', label='saturated: gradient ≈ 0')
axes[1].set_xlabel('$z$')
axes[1].set_ylabel("$\sigma'(z)$")
axes[1].set_title("Sigmoid derivative — gradient channel\nmax 0.25, saturates for |z| > 3")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"σ(0)  = {sigmoid(0):.3f}   (exactly 0.5 — sits on the boundary)")
print(f"σ(3)  = {sigmoid(3):.4f}  σ'(3)  = {sigmoid_deriv(3):.4f}")
print(f"σ(−3) = {sigmoid(-3):.4f}  σ'(−3) = {sigmoid_deriv(-3):.4f}")


### Think about it

- The derivative $\sigma'(z)$ has a maximum of exactly $0.25$.   What does that mean for a gradient that has to pass through ten sigmoid neurons in sequence?
- The shaded red regions mark where $|z| > 3$. What value of weight magnitude $|w|$   would push a typical input ($x \sim 1$) into the saturated region?
- If $w_1 = 0$ and $w_2 = 0$, then $z = b$ for every input. What does the   neuron predict? What happens to the gradient with respect to $w_1$ and $w_2$?
- The sigmoid maps $\mathbb{R} \to (0,1)$. Does that mean the output is a   well-calibrated probability? What additional condition would be required?

In [2]:
np.random.seed(0)

# ── Toy dataset ────────────────────────────────────────────────────────────
# Two clean, well-separated clusters.
# Transits: deep dips, long duration.
# Artifacts: shallow, brief.
n = 40
X_toy = np.vstack([
    np.column_stack([np.random.normal(1.2, 0.12, n),
                     np.random.normal(4.0, 0.45, n)]),
    np.column_stack([np.random.normal(0.2, 0.08, n),
                     np.random.normal(1.0, 0.35, n)]),
])
y_toy = np.array([0]*n + [1]*n)   # 0 = transit,  1 = artifact

# ── Realistic dataset ──────────────────────────────────────────────────────
# Inspired by Kepler planet candidate statistics.
# Classes overlap — no linear boundary can perfectly separate them.
n2 = 100
X_real = np.vstack([
    np.column_stack([np.clip(np.random.normal(0.7, 0.35, n2), 0.05, 3.0),
                     np.clip(np.random.normal(3.5, 1.5,  n2), 0.3, 12.0)]),
    np.column_stack([np.clip(np.random.normal(0.5, 0.40, n2), 0.05, 3.0),
                     np.clip(np.random.normal(2.5, 1.8,  n2), 0.3, 12.0)]),
])
y_real = np.array([0]*n2 + [1]*n2)


# ── Shared plotting function ───────────────────────────────────────────────
def plot_neuron(X_data, y_data, w1, w2, b, xlabel, ylabel):
    z     = X_data @ np.array([w1, w2]) + b
    y_hat = sigmoid(z)
    acc   = np.mean((y_hat > 0.5) == y_data)

    x0 = np.linspace(X_data[:, 0].min() - 0.2, X_data[:, 0].max() + 0.2, 200)
    x1 = np.linspace(X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5, 200)
    xx, yy = np.meshgrid(x0, x1)
    Z = sigmoid(xx * w1 + yy * w2 + b)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.25, vmin=0, vmax=1)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
    ax.scatter(X_data[y_data==0, 0], X_data[y_data==0, 1],
               c='tab:blue', s=35, label='Transit',  zorder=3)
    ax.scatter(X_data[y_data==1, 0], X_data[y_data==1, 1],
               c='tab:red',  s=35, label='Artifact', zorder=3)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'w₁={w1:.1f}  w₂={w2:.1f}  b={b:.1f}    accuracy={acc:.0%}')
    ax.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

---

## Part 1: The toy problem

A neuron maps a feature vector to a single number:

$$y = \sigma(\mathbf{w} \cdot \mathbf{x} + b)$$

- **w**: weights — how much each feature contributes
- **b**: bias — shifts the threshold
- **σ**: activation — sigmoid here, so $y \in (0, 1)$

The **decision boundary** is the line where $\mathbf{w} \cdot \mathbf{x} + b = 0$ — where the neuron outputs exactly 0.5.

The toy data is clean: real exoplanet transits produce deep, long dips in stellar brightness; instrumental artifacts tend to be shallow and brief. Find the boundary that separates them.

*If exoplanets aren't your thing: this is just a 2-feature binary classification problem with two well-separated classes — the ML idea is identical for any such task.*

In [3]:
def plot_toy(w1, w2, b):
    plot_neuron(X_toy, y_toy, w1, w2, b,
                xlabel='Dip depth (%)', ylabel='Dip duration (h)')

interact(
    plot_toy,
    w1=FloatSlider(value=1.0, description='w₁', **kw),
    w2=FloatSlider(value=1.0, description='w₂', **kw),
    b =FloatSlider(value=0.0, description='b',  **kw),
);

interactive(children=(FloatSlider(value=1.0, description='w₁', max=6.0, min=-6.0), FloatSlider(value=1.0, desc…

### Think about it

- What is the highest accuracy you can reach on this data? Is 100% achievable?
- Set $w_1 = 0$. The boundary is now horizontal — the neuron is ignoring one feature entirely. Which one, and does the accuracy suffer?
- Which weight matters more for separating the classes here? What does that tell you about which feature carries more information?
- What happens to the predicted probabilities deep in the blue and red regions as you increase $|w_1|$ and $|w_2|$ together?

---

## Part 2: Activation functions

The activation $\sigma$ is what stops a stack of layers from collapsing to a single linear transformation. Without it, depth buys you nothing.

**The gradient matters as much as the output.** During backpropagation, the gradient of the loss flows through $\sigma'(z)$ at every neuron. Where $\sigma'(z) \approx 0$, learning stops.

In [4]:
def plot_activation(fn):
    x = np.linspace(-5, 5, 400)

    fns = {
        'Linear'    : (x,                           np.ones_like(x)),
        'Sigmoid'   : (sigmoid(x),                  sigmoid(x) * (1 - sigmoid(x))),
        'Tanh'      : (np.tanh(x),                  1 - np.tanh(x)**2),
        'ReLU'      : (np.maximum(0, x),             (x > 0).astype(float)),
        'Leaky ReLU': (np.where(x>0, x, 0.01*x),    np.where(x>0, 1.0, 0.01)),
    }

    y_fn, dy = fns[fn]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    for ax in (ax1, ax2):
        ax.axhline(0, color='grey', lw=0.5)
        ax.axvline(0, color='grey', lw=0.5)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel('z')

    ax1.plot(x, y_fn, 'k', lw=2)
    ax1.set_title(f'σ(z) — {fn}')
    ax1.set_ylabel('σ(z)')

    ax2.plot(x, dy, color='tab:red', lw=2)
    ax2.set_title(f"σ'(z) — {fn}")
    ax2.set_ylabel("σ'(z)")

    plt.tight_layout()
    plt.show()

interact(
    plot_activation,
    fn=ToggleButtons(
        options=['Linear', 'Sigmoid', 'Tanh', 'ReLU', 'Leaky ReLU'],
        value='Sigmoid',
        description='',
    ),
);

interactive(children=(ToggleButtons(description='fn', index=1, options=('Linear', 'Sigmoid', 'Tanh', 'ReLU', '…

### Think about it

- For which activation is $\sigma'(z) = 0$ over exactly half the input domain? What happens to a neuron whose input is always in that half?
- Sigmoid and tanh both saturate. What is the practical difference when used as hidden-layer activations?
- ReLU is not differentiable at $z = 0$. Why does this not cause problems in practice?
- A neuron using ReLU, whose input $z$ is always negative during training, will never update its weights. How would you detect this? How would you fix it?

---

## Part 3: The real problem

Same neuron. Same two features. Now the data comes from a realistic distribution of Kepler planet candidates and known false positives.

The classes overlap. There is no linear boundary that separates them perfectly — you are looking for the **best** boundary, not the right one.

In [5]:
def plot_real(w1, w2, b):
    plot_neuron(X_real, y_real, w1, w2, b,
                xlabel='Dip depth (%)', ylabel='Dip duration (h)')

interact(
    plot_real,
    w1=FloatSlider(value=1.0, description='w₁', **kw),
    w2=FloatSlider(value=1.0, description='w₂', **kw),
    b =FloatSlider(value=0.0, description='b',  **kw),
);

interactive(children=(FloatSlider(value=1.0, description='w₁', max=6.0, min=-6.0), FloatSlider(value=1.0, desc…

### Think about it

- What is the best accuracy you can reach? Why can you not do better?
- Try to construct a boundary with high recall on transits (few missed transits) at the cost of more false positives. What does the boundary look like?
- If a missed transit (false negative) costs more than a wasted follow-up observation (false positive), how should that change where you place the boundary?
- What would you need — more features, a different model — to push accuracy higher?

---

## Part 4: XOR — where one neuron breaks

Parts 1 and 3 were both linear separation problems — one easy, one hard. XOR is categorically different: **no linear boundary can solve it, for any choice of weights**.

| $x_1$ | $x_2$ | XOR |
|--------|--------|-----|
| 0      | 0      | 0   |
| 0      | 1      | 1   |
| 1      | 0      | 1   |
| 1      | 1      | 0   |

The two class-0 points sit at opposite corners. Any line that separates them cuts through the class-1 points. Try.

In [6]:
X_xor = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = np.array([0, 1, 1, 0])

def plot_xor(w1, w2, b):
    z      = X_xor @ np.array([w1, w2]) + b
    y_hat  = (sigmoid(z) > 0.5).astype(int)
    n_corr = int(np.sum(y_hat == y_xor))

    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200),
                         np.linspace(-0.5, 1.5, 200))
    Z = sigmoid(xx*w1 + yy*w2 + b)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.25, vmin=0, vmax=1)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)

    colors = ['tab:blue' if yi == 0 else 'tab:red' for yi in y_xor]
    ax.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=300,
               zorder=3, edgecolors='k', linewidths=1.5)

    for (x0, x1), lbl in zip(X_xor, ['(0,0)=0', '(0,1)=1', '(1,0)=1', '(1,1)=0']):
        ax.annotate(lbl, (x0, x1), xytext=(10, 8),
                    textcoords='offset points', fontsize=9)

    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_aspect('equal')
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.set_title(f'Correct: {n_corr} / 4  —  maximum with one neuron: 3 / 4')
    plt.tight_layout()
    plt.show()

interact(
    plot_xor,
    w1=FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='w₁'),
    w2=FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='w₂'),
    b =FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='b'),
);

interactive(children=(FloatSlider(value=1.0, description='w₁', max=5.0, min=-5.0), FloatSlider(value=1.0, desc…

### Think about it

- Convince yourself the maximum is 3/4. Which three points can you correctly classify, and which one always escapes?
- If you could add a third feature $x_3 = x_1 \cdot x_2$, could one neuron now solve XOR? Why?
- In the transit problem: suppose you added a third class — eclipsing binaries — that mimics transit shape but with a different period distribution. Is that likely to be linearly separable from both transits and artifacts?
- What is the minimum number of neurons you need to solve XOR exactly? Think geometrically.

---

## Part 5 — The loss landscape

The exercises above asked you to *manually* find good weights. But why does gradient descent find them automatically? Because the loss is a smooth surface in weight space — and the gradient always points uphill.

Below: the MSE loss over the toy dataset as a function of $(w_1, w_2)$ (bias fixed at the optimum). The **gradient field** shows the direction of steepest ascent at every point; gradient descent steps in the opposite direction.

Notice:
- For this data the loss surface is bowl-shaped with a single minimum. (Strictly: MSE on a *linear* model is convex; with a sigmoid output the surface can in principle have flat regions, but for well-separated data like this one it stays bowl-shaped in practice.)
- The contours are elliptical because the two features have different scales.   This asymmetry directly causes slow convergence along the shallow axis.
- The arrows far from the minimum are long; near the minimum they shrink —   the gradient naturally decreases as you approach the optimum.

In [ ]:
# ── Part 5: MSE loss surface over (w1, w2) for the toy dataset ───────────────

def mse_surface(X, y, w1_range, w2_range, b=0.0):
    """Compute MSE loss on a grid of (w1, w2) values."""
    W1, W2 = np.meshgrid(w1_range, w2_range)
    loss = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            z = X[:, 0] * W1[i, j] + X[:, 1] * W2[i, j] + b
            p = sigmoid(z)
            loss[i, j] = np.mean((p - y) ** 2)
    return W1, W2, loss

w1_range = np.linspace(-3, 6, 80)
w2_range = np.linspace(-1, 4, 80)

# Use toy dataset — well-separated, clean gradient structure
W1g, W2g, L = mse_surface(X_toy, y_toy, w1_range, w2_range, b=0.0)

# ── Gradient of MSE w.r.t. (w1, w2) — finite differences ─────────────────
dw1 = np.gradient(L, w1_range, axis=1)
dw2 = np.gradient(L, w2_range, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: contour map
cf = axes[0].contourf(W1g, W2g, L, levels=30, cmap='viridis')
axes[0].contour(W1g,  W2g, L, levels=10, colors='white', linewidths=0.5, alpha=0.4)
plt.colorbar(cf, ax=axes[0], label='MSE loss')

# Gradient field (subsample for readability)
skip = 8
axes[0].quiver(W1g[::skip, ::skip], W2g[::skip, ::skip],
               dw1[::skip, ::skip], dw2[::skip, ::skip],
               color='white', alpha=0.7, scale=12, width=0.004)

axes[0].set_xlabel('$w_1$  (dip-depth weight)')
axes[0].set_ylabel('$w_2$  (dip-duration weight)')
axes[0].set_title('MSE loss surface — toy dataset\nArrows point uphill (gradient)')

# Right: 3D surface
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401
ax3d = fig.add_subplot(1, 2, 2, projection='3d')
ax3d.plot_surface(W1g, W2g, L, cmap='viridis', alpha=0.85, linewidth=0)
ax3d.set_xlabel('$w_1$'); ax3d.set_ylabel('$w_2$'); ax3d.set_zlabel('MSE')
ax3d.set_title('Same surface in 3D')
ax3d.view_init(elev=30, azim=-60)

plt.tight_layout()
plt.show()

# Mark the approximate minimum
min_idx = np.unravel_index(L.argmin(), L.shape)
print(f'Approximate minimum near w1={W1g[min_idx]:.2f}, w2={W2g[min_idx]:.2f}')
print(f'Minimum MSE ≈ {L.min():.4f}')


### Think about it

- The contour ellipses are tilted. What does the orientation tell you about the   correlation between $w_1$ and $w_2$ at the minimum?
- Follow a gradient arrow from a point far from the minimum. Does it point directly   toward the minimum, or at an angle? Why might it miss?
- For the realistic dataset (overlapping classes), would the loss surface still have a   single minimum? Would it be flatter or sharper at the bottom?
- What would the loss surface look like for a **two-layer** network?   (Hint: is it still a bowl?)

---

## Exercise: implement `neuron_forward`

Implement the single neuron forward pass in NumPy. Your function must handle both a single input vector and a batch of vectors.

In [ ]:
def neuron_forward(x, w, b, activation='sigmoid'):
    """
    Single neuron forward pass.

    Parameters
    ----------
    x          : array, shape (n_features,) or (n_samples, n_features)
    w          : array, shape (n_features,)
    b          : float
    activation : 'sigmoid' | 'tanh' | 'relu' | 'linear'

    Returns
    -------
    y : array, shape () or (n_samples,)
    """
    # Step 1: weighted sum + bias  →  z = w · x + b
    # Step 2: apply the activation function
    # Hint: np.dot handles both single vectors and batches when shapes align
    raise NotImplementedError("Fill in neuron_forward")

In [ ]:
import torch

x_t = np.array([0.75, -0.6])
w_t = np.array([2.0, -1.5])
b_t = 0.3

out = neuron_forward(x_t, w_t, b_t, activation='sigmoid')
ref = torch.sigmoid(
    torch.tensor(x_t @ w_t + b_t, dtype=torch.float32)
).item()

print(f'Your output : {float(out):.6f}')
print(f'PyTorch ref : {ref:.6f}')
assert abs(float(out) - ref) < 1e-5, 'Mismatch — check your implementation'
print('\u2713  Correct')

# batch shape test
X_batch = np.random.randn(10, 2)
out_batch = neuron_forward(X_batch, w_t, b_t, activation='relu')
assert out_batch.shape == (10,), f'Expected (10,), got {out_batch.shape}'
print('\u2713  Batch shape correct')

---

## Going further

The exercise above implemented the forward pass for a fixed architecture.
The two challenges below push in different directions: one goes deeper into
what a single neuron can and cannot do; the other points toward what comes next.

### Part A — Go deeper

Real Kepler planet-candidate vetting does not use two features. The NASA Robovetter
pipeline uses dozens: transit depth, duration, centroid shift, secondary eclipse depth,
odd/even depth difference, and more. Each additional feature is another dimension in which
the boundary lives — but the neuron still draws a hyperplane.

**Challenge:** Add a third feature — secondary eclipse depth — to the toy dataset
and explore how the decision boundary changes.

- Extend `X_toy` with a third column. For real transits, secondary eclipses are tiny
  (depth ≈ 0); for background eclipsing binaries misidentified as artifacts, they can be
  substantial (depth ≈ 0.3–0.8). Generate plausible synthetic values for both classes.
- Add a third weight slider $w_3$ to the `interact()` call.
- Does the third feature improve classification accuracy on the toy data? On the realistic data?

*Suggested approach:* Absorb the third feature's contribution into the bias term for
visualisation — fix $w_3$ and treat $b' = b + w_3 \cdot \bar{x}_3$ as an effective bias,
so the existing 2D scatter plot still makes sense without adding a 3D plot.

### Part B — Lead forward

You proved that no single neuron can solve XOR — the maximum is 3 out of 4 correct
for any choice of weights. But geometrically, *two* lines can carve the plane into
regions that correctly separate all four XOR points.

**Challenge:** Design a 2 → 2 → 1 network by hand that solves XOR exactly.
Write out weight matrices $W_1 \in \mathbb{R}^{2 \times 2}$,
$W_2 \in \mathbb{R}^{1 \times 2}$ and biases, then verify using `neuron_forward`:
apply it once per layer, and confirm all four XOR inputs produce the correct output.

*Suggested approach:* Think about what each hidden neuron should detect.

- Hidden neuron 1: fires when **at least one** input is 1 — this is OR
- Hidden neuron 2: fires when **both** inputs are 1 — this is AND
- Output neuron: fires when neuron 1 is on **and** neuron 2 is off

Write weights that implement each of those three operations, then verify.
In the next notebook you will see how gradient descent finds these weights automatically —
without you having to think geometrically at all.

---

### References

| | |
|---|---|
| **Video** | 3Blue1Brown — *But what is a neural network?* [youtube.com/watch?v=aircAruvnKk](https://www.youtube.com/watch?v=aircAruvnKk) |
| **Primary** | McCulloch & Pitts (1943) — *A logical calculus of the ideas immanent in nervous activity.* Bulletin of Mathematical Biophysics 5, 115–133 |
| **Primary** | Rosenblatt (1958) — *The Perceptron: a probabilistic model for information storage and organization in the brain.* Psychological Review 65(6), 386–408 |
| **Blog** | Karpathy — *Hacker's guide to Neural Networks* [karpathy.github.io/neuralnets/](http://karpathy.github.io/neuralnets/) |
| **Interactive** | TensorFlow Playground — draw decision boundaries in the browser [playground.tensorflow.org](https://playground.tensorflow.org) |

---

> **Try the exercise yourself first.** The solution is below — scroll past it if you haven't attempted the exercise.


---

## Solution — `neuron_forward`

The forward pass has two steps: compute the weighted sum $z = \mathbf{w} \cdot \mathbf{x} + b$, then apply the activation. `np.dot` handles both a single vector and a batch automatically because NumPy broadcasts matrix multiplication when shapes align.

In [ ]:
def neuron_forward(x, w, b, activation='sigmoid'):
    # Step 1: weighted sum.  np.dot works for both shapes:
    #   x.shape = (n_features,)       → z is a scalar
    #   x.shape = (n_samples, n_features) → z is (n_samples,)
    z = np.dot(x, w) + b

    # Step 2: activation
    if activation == 'sigmoid':
        return 1 / (1 + np.exp(-z))
    elif activation == 'tanh':
        return np.tanh(z)
    elif activation == 'relu':
        return np.maximum(0, z)
    elif activation == 'linear':
        return z
    else:
        raise ValueError(f"Unknown activation: {activation}")

# ── What this connects back to ────────────────────────────────────────────
# Part 1 slider: you were adjusting w and b manually to move the boundary z = 0.
# This function computes p = sigma(z) for every data point simultaneously.
# Parts 3/4 showed that sigma(z) saturates for large |z| — visible here as the
# gradient of the sigmoid approaching zero away from z = 0.
